# Chapter 6 - Algorithm Chains and Pipelines

**Book:** Introduction to Machine Learning with Python  
**Authors:** Andreas C. Müller & Sarah Guido

Chapter ini membahas cara menggabungkan beberapa tahap machine
learning menjadi satu workflow menggunakan pipelines.

Pipeline dapat menggabungkan preprocessing, feature selection,
dan machine learning model dalam satu proses.

## 1. Chapter Overview

Chapter ini membahas:

1. Data Leakage
2. Preprocessing dan Pipelines
3. Building Pipelines
4. Using Pipelines with Grid Search
5. Feature Extraction
6. Pipeline untuk Classification
7. Pipeline untuk Regression
8. Evaluasi Pipeline

## 2. Why Use Pipelines?

Dalam machine learning, proses biasanya terdiri dari beberapa tahap:

Data
↓
Preprocessing
↓
Feature Selection
↓
Model
↓
Prediction

Tanpa pipeline, setiap tahap harus dilakukan secara manual.

Pipeline memungkinkan seluruh proses tersebut digabung menjadi
satu objek sehingga lebih mudah digunakan dan mengurangi risiko
kesalahan.

## 3. Data Leakage

Data leakage terjadi ketika informasi dari data testing masuk
ke dalam proses training.

Contohnya:

1. Melakukan scaling pada seluruh dataset.
2. Baru membagi data menjadi training dan testing.

Hal ini menyebabkan informasi dari testing ikut digunakan
untuk menentukan preprocessing.

Cara yang lebih aman adalah:

Training Data
↓
fit preprocessing
↓
transform training

Testing Data
↓
transform menggunakan preprocessing training

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()

X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (426, 30)
Testing : (143, 30)


## 5. Building a Simple Pipeline

Pipeline dapat menggabungkan preprocessing dan model.

Contoh:

StandardScaler
↓
Logistic Regression

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

pipeline.fit(
    X_train,
    y_train
)

print(
    "Test accuracy:",
    pipeline.score(X_test, y_test)
)

Test accuracy: 0.986013986013986


## 6. Why Pipelines Are Useful

Pipeline memastikan setiap tahap preprocessing dilakukan
dengan benar.

Saat:

pipeline.fit()

scaler akan melakukan fit hanya pada data training.

Saat:

pipeline.predict()

scaler menggunakan parameter yang sudah diperoleh dari
training untuk melakukan transformasi pada data baru.

Dengan demikian, risiko data leakage dapat dikurangi.

## 7. Pipeline with Grid Search

Pipeline dapat digabung dengan GridSearchCV.

Hal ini memungkinkan kita mencari hyperparameter sekaligus
melakukan preprocessing.

Parameter dalam pipeline ditulis menggunakan format:

nama_step__nama_parameter

Contoh:

model__C

In [3]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5
)

grid.fit(
    X_train,
    y_train
)

print("Best parameter:")
print(grid.best_params_)

print("\nBest CV score:")
print(grid.best_score_)

print("\nTest score:")
print(grid.score(X_test, y_test))

Best parameter:
{'model__C': 0.1}

Best CV score:
0.9765253077975377

Test score:
0.9790209790209791


## 8. Pipeline with Feature Selection

Pipeline juga dapat menggabungkan:

Scaling
↓
Feature Selection
↓
Model

Dengan cara ini, proses pemilihan fitur dilakukan
secara otomatis di dalam pipeline.

In [4]:
from sklearn.feature_selection import SelectKBest, f_classif

pipeline_selection = Pipeline([
    ("scaler", StandardScaler()),
    ("selection", SelectKBest(
        score_func=f_classif,
        k=10
    )),
    ("model", LogisticRegression(max_iter=5000))
])

pipeline_selection.fit(
    X_train,
    y_train
)

print(
    "Test accuracy:",
    pipeline_selection.score(
        X_test,
        y_test
    )
)

Test accuracy: 0.951048951048951


## 9. Pipeline with Feature Selection and Grid Search

Pipeline dapat digunakan untuk mencari:

- jumlah fitur terbaik
- hyperparameter model

secara bersamaan.

In [5]:
param_grid = {
    "selection__k": [5, 10, 15, 20],
    "model__C": [0.01, 0.1, 1, 10]
}

grid_pipeline = GridSearchCV(
    pipeline_selection,
    param_grid,
    cv=5
)

grid_pipeline.fit(
    X_train,
    y_train
)

print("Best parameters:")
print(grid_pipeline.best_params_)

print("\nBest CV score:")
print(grid_pipeline.best_score_)

print("\nTest score:")
print(grid_pipeline.score(X_test, y_test))

Best parameters:
{'model__C': 0.1, 'selection__k': 20}

Best CV score:
0.976497948016416

Test score:
0.9790209790209791


## 10. Pipeline for Regression

Pipeline tidak hanya digunakan untuk classification.

Pipeline juga dapat digunakan untuk regression.

Contoh:

Scaling
↓
Ridge Regression

In [6]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge

diabetes = load_diabetes()

X_reg = diabetes.data
y_reg = diabetes.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42
)

reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

reg_pipeline.fit(
    X_train_reg,
    y_train_reg
)

print(
    "R² score:",
    reg_pipeline.score(
        X_test_reg,
        y_test_reg
    )
)

R² score: 0.48589618270899804


## 11. Pipeline with Cross-Validation

Pipeline juga dapat langsung digunakan
dengan cross-validation.

Hal ini membuat preprocessing dilakukan
secara terpisah pada setiap training fold.

In [7]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5
)

print("Scores:", scores)
print("Mean:", scores.mean())

Scores: [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
Mean: 0.9806862288464524


## 12. Pipeline with Categorical Data

Pipeline juga dapat digabung dengan ColumnTransformer.

Contohnya:

Categorical Data
→ One-Hot Encoding

Numerical Data
→ Scaling

Kemudian hasilnya diberikan kepada model.

In [8]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

data = pd.DataFrame({
    "city": [
        "Bandung", "Jakarta", "Surabaya",
        "Bandung", "Jakarta", "Surabaya",
        "Bandung", "Jakarta", "Surabaya",
        "Bandung", "Jakarta", "Surabaya"
    ],
    "age": [
        20, 25, 30,
        22, 27, 35,
        24, 29, 31,
        21, 26, 33
    ],
    "income": [
        4, 7, 8,
        5, 6, 9,
        5, 7, 8,
        4, 6, 9
    ],
    "target": [
        0, 1, 1,
        0, 1, 1,
        0, 1, 1,
        0, 1, 1
    ]
})

X_cat = data[
    ["city", "age", "income"]
]

y_cat = data["target"]

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.25,
    random_state=42,
    stratify=y_cat
)

In [9]:
preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        ["city"]
    ),
    (
        "numeric",
        StandardScaler(),
        ["age", "income"]
    )
])

complete_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

complete_pipeline.fit(
    X_train_cat,
    y_train_cat
)

print(
    "Test accuracy:",
    complete_pipeline.score(
        X_test_cat,
        y_test_cat
    )
)

Test accuracy: 1.0


In [10]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

complete_grid = GridSearchCV(
    complete_pipeline,
    param_grid,
    cv=5
)

complete_grid.fit(
    X_train_cat,
    y_train_cat
)

print("Best parameter:")
print(complete_grid.best_params_)

print("\nBest CV score:")
print(complete_grid.best_score_)

print("\nTest score:")
print(
    complete_grid.score(
        X_test_cat,
        y_test_cat
    )
)

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Best parameter:
{'model__C': 1}

Best CV score:
1.0

Test score:
1.0


## 15. Results and Analysis

Pipeline memungkinkan beberapa tahap machine learning
digabung menjadi satu workflow.

Dalam eksperimen ini:

- StandardScaler digunakan untuk preprocessing.
- SelectKBest digunakan untuk feature selection.
- Logistic Regression digunakan sebagai model.
- GridSearchCV digunakan untuk mencari hyperparameter.
- Cross-validation digunakan untuk mengevaluasi pipeline.

Pipeline juga membantu mengurangi risiko data leakage karena
proses preprocessing dilakukan berdasarkan data training
pada setiap proses evaluasi.

## 16. Comparison

| Pendekatan | Fungsi |
|---|---|
| Train-Test Split | Membagi data |
| Pipeline | Menggabungkan beberapa tahap |
| ColumnTransformer | Preprocessing beberapa tipe fitur |
| Feature Selection | Memilih fitur |
| GridSearchCV | Mencari hyperparameter |
| Cross-Validation | Mengevaluasi model |

## 17. Chapter Summary

Chapter ini membahas algorithm chains dan pipelines.

Hal penting yang dipelajari:

- Pipeline menggabungkan beberapa tahap machine learning.
- Pipeline membantu mencegah data leakage.
- Preprocessing dapat dimasukkan ke dalam pipeline.
- Feature selection dapat dimasukkan ke dalam pipeline.
- Pipeline dapat digunakan bersama GridSearchCV.
- Pipeline dapat digunakan untuk classification maupun regression.
- ColumnTransformer dapat digunakan untuk menangani fitur
  numerik dan kategorikal secara bersamaan.